# Test LGP on FlappyBird with 12-Feature Vector

This notebook tests whether LGP can solve the simpler FlappyBird problem using 12 scalar features instead of image observations.

The 12 features are:
1. last pipe's horizontal position
2. last top pipe's vertical position
3. last bottom pipe's vertical position
4. next pipe's horizontal position
5. next top pipe's vertical position
6. next bottom pipe's vertical position
7. next next pipe's horizontal position
8. next next top pipe's vertical position
9. next next bottom pipe's vertical position
10. player's vertical position
11. player's vertical velocity
12. player's rotation


In [11]:
# Import required libraries
import numpy as np
import os

# Check if flappy-bird-gymnasium is installed
try:
    import flappy_bird_gymnasium
    print("✓ flappy-bird-gymnasium imported successfully")
except ImportError:
    print("✗ flappy-bird-gymnasium not found. Install with: pip install flappy-bird-gymnasium")
    raise

import gymnasium as gym
from scipy.special import expit

from memory_system import MemoryConfig, MemoryBank, MemoryType
from evaluator import FlappyBirdSimpleEvaluator, FlappyBirdSimpleEvaluatorConfig
from instruction_set import InstructionSet
from individual import Individual
from operation import AUTOML_ALL_OPS

print("✓ All imports successful!")


✓ flappy-bird-gymnasium imported successfully
✓ All imports successful!


## Test Environment Setup

First, let's verify the environment works and check the observation format.


In [12]:
# Test the environment with 12-feature vector (use_lidar=False)
print("Testing FlappyBird-v0 environment with 12-feature vector...")
print()

env = gym.make("FlappyBird-v0", render_mode=None, use_lidar=False)
obs, info = env.reset(seed=42)

print(f"Observation shape: {obs.shape}")
print(f"Observation type: {type(obs)}")
print(f"Observation: {obs}")
print()
print(f"Expected: 12 features")
print(f"Got: {len(obs) if hasattr(obs, '__len__') else 'N/A'} features")

if len(obs) == 12:
    print("✓ Observation format is correct!")
else:
    print(f"⚠ Warning: Expected 12 features, got {len(obs)}")

# Test a few steps
for i in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step {i+1}: action={action}, reward={reward:.3f}, terminated={terminated}")
    if terminated or truncated:
        obs, info = env.reset()

env.close()
print("\n✓ Environment test complete!")


Testing FlappyBird-v0 environment with 12-feature vector...

Observation shape: (12,)
Observation type: <class 'numpy.ndarray'>
Observation: [ 1.         0.1953125  0.390625   1.         0.         1.
  1.         0.         1.         0.4765625 -0.9        0.5      ]

Expected: 12 features
Got: 12 features
✓ Observation format is correct!
Step 1: action=1, reward=0.100, terminated=False
Step 2: action=1, reward=0.100, terminated=False
Step 3: action=0, reward=0.100, terminated=False

✓ Environment test complete!


## Create Memory Configuration

Set up memory for 12 scalar observations.


In [13]:
# Memory configuration for 12-feature vector FlappyBird
memory_cfg = MemoryConfig(
    n_scalar=8,          # Working scalar registers
    n_vector=8,          # Working vector registers
    n_matrix=4,          # Working matrix registers (for CV operations if needed)
    n_obs_scalar=12,     # 12 scalar observation registers for features
    n_obs_vector=1,      # 1 vector observation register (12 elements)
    n_obs_matrix=1,      # 1 matrix observation register (12x12)
    vector_size=12,      # Vector size matches number of features
    matrix_shape=(12, 12),  # Matrix shape for tiled observations
)

print("Memory Configuration:")
print(f"  Scalar observations: {memory_cfg.n_obs_scalar}")
print(f"  Vector observations: {memory_cfg.n_obs_vector} (size {memory_cfg.vector_size})")
print(f"  Matrix observations: {memory_cfg.n_obs_matrix} (shape {memory_cfg.matrix_shape})")
print("\n✓ Memory configuration created!")


Memory Configuration:
  Scalar observations: 12
  Vector observations: 1 (size 12)
  Matrix observations: 1 (shape (12, 12))

✓ Memory configuration created!


## Create Evaluator

Set up the FlappyBirdSimpleEvaluator with 12-feature vector observations.


In [14]:
# Create evaluator configuration
evaluator_config = FlappyBirdSimpleEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=5,           # Number of episodes per evaluation
    max_steps=500,        # Maximum steps per episode
    output_register=0,    # Read action from scalar register 0
    render_mode=None,     # No rendering for speed
    use_lidar=False,      # Use 12-feature vector (not LIDAR)
    rng_seed=42,          # Random seed for reproducibility
    output_registers=[(MemoryType.SCALAR, 0)],
)

# Create evaluator
evaluator = FlappyBirdSimpleEvaluator(config=evaluator_config)

print("Evaluator Configuration:")
print(f"  Environment: {evaluator_config.env_id}")
print(f"  Episodes: {evaluator_config.episodes}")
print(f"  Max steps: {evaluator_config.max_steps}")
print(f"  Output register: {evaluator_config.output_register}")
print(f"  Use LIDAR: {evaluator_config.use_lidar}")
print("\n✓ Evaluator created successfully!")


Evaluator Configuration:
  Environment: FlappyBird-v0
  Episodes: 5
  Max steps: 500
  Output register: 0
  Use LIDAR: False

✓ Evaluator created successfully!


## Test with Random Individual

Create a random individual and test it to see baseline performance.


In [15]:
# Create instruction set
rng = np.random.default_rng(42)

# Create template memory from config
template_memory = MemoryBank(
    n_scalar=memory_cfg.n_scalar,
    n_vector=memory_cfg.n_vector,
    n_matrix=memory_cfg.n_matrix,
    n_obs_scalar=memory_cfg.n_obs_scalar,
    n_obs_vector=memory_cfg.n_obs_vector,
    n_obs_matrix=memory_cfg.n_obs_matrix,
    vector_size=memory_cfg.vector_size,
    matrix_shape=memory_cfg.matrix_shape,
    rng=rng
)

operations = [op() for op in AUTOML_ALL_OPS]
instruction_set = InstructionSet(operations, template_memory)

print(f"Instruction set created with {len(operations)} operations")
print()

# Create a random individual
print("Creating random individual...")
random_individual = Individual.random(
    instruction_set=instruction_set,
    memory_config=memory_cfg,
    program_length=20,  # Short program for testing
    rng=rng
)

print(f"✓ Random individual created with {len(random_individual.program)} instructions")
print()

# Evaluate the random individual
print("Evaluating random individual...")
fitness = evaluator.evaluate(random_individual)

print(f"\nRandom Individual Fitness: {fitness:.4f}")
print(f"  (Average reward across {evaluator_config.episodes} episodes)")


Instruction set created with 64 operations

Creating random individual...
✓ Random individual created with 20 instructions

Evaluating random individual...

Random Individual Fitness: -9.3000
  (Average reward across 5 episodes)


## Evolution Setup

Now let's set up a complete evolution loop with:
- Population size: 100
- Elites: 20
- Generations: 100


In [16]:
# Import evolution components
from population import Population, PopulationConfig
from operators import GeneticOperators
from evolution_engine import EvolutionEngine, EvolutionConfig

print("✓ Evolution components imported!")


✓ Evolution components imported!


In [17]:
# Create instruction set for evolution
# Use all AutoML operations (scalar operations work well with feature vectors)
rng_evolution = np.random.default_rng(42)

# Create template memory from config
template_memory_evolution = MemoryBank(
    n_scalar=memory_cfg.n_scalar,
    n_vector=memory_cfg.n_vector,
    n_matrix=memory_cfg.n_matrix,
    n_obs_scalar=memory_cfg.n_obs_scalar,
    n_obs_vector=memory_cfg.n_obs_vector,
    n_obs_matrix=memory_cfg.n_obs_matrix,
    vector_size=memory_cfg.vector_size,
    matrix_shape=memory_cfg.matrix_shape,
    rng=rng_evolution
)

operations_evolution = [op() for op in AUTOML_ALL_OPS]
instruction_set_evolution = InstructionSet(
    operations_evolution, 
    template_memory_evolution
)

print(f"Instruction set created with {len(operations_evolution)} operations")
print("\n✓ Instruction set ready for evolution!")


Instruction set created with 64 operations

✓ Instruction set ready for evolution!


In [18]:
# Create genetic operators
genetic_operators = GeneticOperators(instruction_set_evolution, rng=rng_evolution)

print("✓ Genetic operators created!")


✓ Genetic operators created!


In [19]:
# Create population configuration
population_config = PopulationConfig(
    size=100,              # Population size
    program_length=(10, 50),  # Initial program length range
    elitism=20,            # Number of elites to preserve
    max_program_length=100,  # Maximum program length
)

print("Population Configuration:")
print(f"  Size: {population_config.size}")
print(f"  Program length range: {population_config.program_length}")
print(f"  Elitism: {population_config.elitism}")
print(f"  Max program length: {population_config.max_program_length}")
print("\n✓ Population configuration created!")


Population Configuration:
  Size: 100
  Program length range: (10, 50)
  Elitism: 20
  Max program length: 100

✓ Population configuration created!


In [20]:
# Create and initialize population
print("Creating and initializing population...")
population = Population(
    config=population_config,
    instruction_set=instruction_set_evolution,
    memory_config=memory_cfg,
    operators=genetic_operators,
    rng=rng_evolution
)

# Initialize with random individuals and mutate constants
population.initialize_random(mutate_constants=True)

print(f"✓ Population initialized with {len(population.individuals)} individuals")
print(f"  Average program length: {np.mean([len(ind.program) for ind in population.individuals]):.1f}")


Creating and initializing population...
✓ Population initialized with 100 individuals
  Average program length: 31.5


In [21]:
# Create evaluator configuration for evolution
evolution_evaluator_config = FlappyBirdSimpleEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=5,           # Number of episodes per evaluation
    max_steps=500,        # Maximum steps per episode
    output_register=0,    # Read action from scalar register 0
    render_mode=None,     # No rendering for speed
    use_lidar=False,      # Use 12-feature vector
    rng_seed=42,          # Random seed for reproducibility
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,             # Sequential processing (adjust for parallel)
)

# Create evaluator for evolution
evolution_evaluator = FlappyBirdSimpleEvaluator(config=evolution_evaluator_config)

print("Evolution Evaluator Configuration:")
print(f"  Episodes per evaluation: {evolution_evaluator_config.episodes}")
print(f"  Max steps per episode: {evolution_evaluator_config.max_steps}")
print(f"  Output register: {evolution_evaluator_config.output_register}")
print("\n✓ Evolution evaluator created!")


Evolution Evaluator Configuration:
  Episodes per evaluation: 5
  Max steps per episode: 500
  Output register: 0

✓ Evolution evaluator created!


In [22]:
# Create evolution configuration
evolution_config = EvolutionConfig(
    max_generations=100,         # Number of generations
    mutation_threshold=0.9,      # Probability of mutation
    constant_mutation_rate=0.1,  # Probability of mutating constants
    crossover_threshold=0.9,     # Probability of crossover
    verbose=True,                # Print generation statistics
    checkpoint_dir="checkpoints_flappy_simple",  # Checkpoint directory
    checkpoint_every=10,         # Save checkpoint every 10 generations
    stats_log_path="stats_flappy_simple.csv",    # Statistics log file
)

print("Evolution Configuration:")
print(f"  Max generations: {evolution_config.max_generations}")
print(f"  Mutation threshold: {evolution_config.mutation_threshold}")
print(f"  Constant mutation rate: {evolution_config.constant_mutation_rate}")
print(f"  Crossover threshold: {evolution_config.crossover_threshold}")
print(f"  Checkpoint directory: {evolution_config.checkpoint_dir}")
print(f"  Stats log: {evolution_config.stats_log_path}")
print("\n✓ Evolution configuration created!")


Evolution Configuration:
  Max generations: 100
  Mutation threshold: 0.9
  Constant mutation rate: 0.1
  Crossover threshold: 0.9
  Checkpoint directory: checkpoints_flappy_simple
  Stats log: stats_flappy_simple.csv

✓ Evolution configuration created!


In [23]:
# Create evolution engine
evolution_engine = EvolutionEngine(
    population=population,
    operators=genetic_operators,
    evaluator=evolution_evaluator,
    config=evolution_config,
    rng=rng_evolution,
)

print("✓ Evolution engine created!")
print("\nReady to run evolution!")


Statistics logging: stats_flappy_simple.csv
✓ Evolution engine created!

Ready to run evolution!


## Run Evolution

This will run the complete evolution loop for 100 generations. This may take a while depending on your system.


In [ ]:
# Run the evolution loop
print("="*80)
print("STARTING EVOLUTION")
print("="*80)
print()
print(f"Population size: {population_config.size}")
print(f"Elites: {population_config.elitism}")
print(f"Generations: {evolution_config.max_generations}")
print(f"Episodes per evaluation: {evolution_evaluator_config.episodes}")
print()
print("Starting evolution loop...")
print("="*80)
print()

# Run evolution
final_population = evolution_engine.run()

print()
print("="*80)
print("EVOLUTION COMPLETE")
print("="*80)
print()

# Close evaluator
evolution_evaluator.close()


STARTING EVOLUTION

Population size: 100
Elites: 20
Generations: 100
Episodes per evaluation: 5

Starting evolution loop...

Evaluated 10/100 individuals
Evaluated 20/100 individuals
Evaluated 30/100 individuals
Evaluated 40/100 individuals
Evaluated 50/100 individuals
Evaluated 60/100 individuals
Evaluated 70/100 individuals
Evaluated 80/100 individuals
Evaluated 90/100 individuals
Evaluated 100/100 individuals
  → Initialized best_ever (fitness: 4.0000, gen: 0)
Best agent: fitness=4.0000, effective_code_rate=0.045 (1/22)
Checkpoint saved: gen=0, fitness=4.0000 -> checkpoints_flappy_simple/gen_0000.pkl

=== Generation 0 ===
Generation 0 | Population size 100
Min: -9.300, Mean: -6.210, Max: 4.000, Std: 5.091
Length mean 31.5, std 11.1
Best ever fitness 4.000 at generation 0
Evaluated 10/100 individuals
Evaluated 20/100 individuals
Evaluated 30/100 individuals
Evaluated 40/100 individuals
Evaluated 50/100 individuals
Evaluated 60/100 individuals
Evaluated 70/100 individuals
Evaluated 80

/Users/xavierhillroy/Desktop/Super/Brains/Academic/McMaster/Courses/CAS739/Project/LGP_VISON/operation.py:1369: RuntimeWarning: overflow encountered in multiply
  return (s * m).astype(np.float32)


Evaluated 100/100 individuals
Best agent: fitness=5.0200, effective_code_rate=0.037 (1/27)

=== Generation 9 ===
Generation 9 | Population size 100
Min: -9.300, Mean: -0.416, Max: 5.020, Std: 5.837
Length mean 24.8, std 9.2
Best ever fitness 5.020 at generation 7
Evaluated 10/100 individuals
Evaluated 20/100 individuals
Evaluated 30/100 individuals
Evaluated 40/100 individuals
Evaluated 50/100 individuals
Evaluated 60/100 individuals
Evaluated 70/100 individuals
Evaluated 80/100 individuals
Evaluated 90/100 individuals
Evaluated 100/100 individuals
Best agent: fitness=5.0200, effective_code_rate=0.037 (1/27)
Checkpoint saved: gen=10, fitness=5.0200 -> checkpoints_flappy_simple/gen_0010.pkl

=== Generation 10 ===
Generation 10 | Population size 100
Min: -9.300, Mean: -0.942, Max: 5.020, Std: 5.876
Length mean 25.4, std 9.7
Best ever fitness 5.020 at generation 7
Evaluated 10/100 individuals
Evaluated 20/100 individuals
Evaluated 30/100 individuals
Evaluated 40/100 individuals
Evaluated 

## Analyze Results

Display the final population statistics and best agent information.


In [ ]:
# Display final population summary
final_population.print_summary()

print()
print("="*80)
print("BEST AGENT INFORMATION")
print("="*80)
print()

if final_population.best_ever is not None:
    best_agent = final_population.best_ever
    print(f"Best fitness: {best_agent.fitness:.4f}")
    print(f"Best fitness generation: {final_population.best_ever_generation}")
    print(f"Program length: {len(best_agent.program)}")
    print(f"Agent ID: {best_agent.id}")
    print(f"Age: {best_agent.age}")
    
    # Calculate effective code rate
    if hasattr(evolution_evaluator, 'output_registers'):
        output_regs = evolution_evaluator.output_registers
    else:
        output_regs = [(MemoryType.SCALAR, 0)]
    
    effective_program = best_agent.get_effective_program(output_regs)
    effective_code_rate = len(effective_program) / len(best_agent.program) if len(best_agent.program) > 0 else 0
    
    print(f"Effective program length: {len(effective_program)}")
    print(f"Effective code rate: {effective_code_rate:.3f} ({effective_code_rate*100:.1f}%)")
else:
    print("No best agent found!")


## Visualize Best Agent

Watch the best agent play FlappyBird.


In [ ]:
# Optional: Visualize the best agent playing
if final_population.best_ever is not None:
    print("Creating visualizer evaluator...")
    
    # Create evaluator with human rendering
    visualizer_config = FlappyBirdSimpleEvaluatorConfig(
        env_id="FlappyBird-v0",
        episodes=3,
        max_steps=500,
        output_register=0,
        render_mode="human",  # Human rendering to see the game
        use_lidar=False,
        rng_seed=42,
        output_registers=[(MemoryType.SCALAR, 0)],
    )
    
    visualizer = FlappyBirdSimpleEvaluator(config=visualizer_config)
    
    best_agent = final_population.best_ever
    print(f"\nRunning best agent (fitness: {best_agent.fitness:.4f})...")
    print("Close the game window to continue.\n")
    
    # Run a few episodes
    for episode_idx in range(visualizer_config.episodes):
        reward = visualizer._evaluate_episode(best_agent, episode_idx)
        print(f"Episode {episode_idx + 1}: Reward = {reward:.4f}")
    
    visualizer.close()
    print("\n✓ Visualization complete!")
else:
    print("No best agent to visualize!")


## Summary

This notebook demonstrates:
1. ✓ Environment setup with 12-feature vector observations
2. ✓ Memory configuration for 12 scalar features
3. ✓ FlappyBirdSimpleEvaluator setup
4. ✓ Testing with random individuals

**Next Steps:**
- Use this evaluator with your evolution engine to train agents
- The 12-feature vector should be much simpler than image-based observations
- Expect faster training and potentially better results
